In [30]:
import json

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

from torch.utils.data import DataLoader, random_split

In [4]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset_full = torchvision.datasets.MNIST(
    root=".",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root=".",
    train=False,
    download=True,
    transform=transform
)
# torchvision.datasets.KMNIST

print("Train full:", len(train_dataset_full))
print("Test:", len(test_dataset))

100.0%
100.0%
100.0%
100.0%

Train full: 60000
Test: 10000


In [9]:
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=generator
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Train: 48000
Val: 12000


In [10]:
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
x_batch, y_batch = next(iter(train_loader))

print("x.shape:", x_batch.shape)
print("y.shape:", y_batch.shape)
print("Min/Max:", x_batch.min().item(), x_batch.max().item())
print("Batch size", BATCH_SIZE)

x.shape: torch.Size([128, 1, 28, 28])
y.shape: torch.Size([128])
Min/Max: 0.0 1.0
Batch size 128


In [13]:
class MLP(nn.Module):
    def __init__(self, input_dim=28*28, hidden_dims=[256], 
                 dropout=0.0, batchnorm=False):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
                
            layers.append(nn.ReLU())
            
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            
            prev_dim = h
        
        layers.append(nn.Linear(prev_dim, 10))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        x = torch.flatten(x, 1)
        return self.model(x)


In [14]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            
            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    
    return total_loss / total, correct / total

In [15]:
def run_experiment(config, epochs=15, early_stopping=False, patience=3):
    model = MLP(
        hidden_dims=config["hidden_dims"],
        dropout=config.get("dropout", 0.0),
        batchnorm=config.get("batchnorm", False)
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }
    
    best_val_acc = 0
    best_epoch = 0
    best_weights = None
    patience_counter = 0
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_weights = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if early_stopping and patience_counter >= patience:
            break
    
    return {
        "config": config,
        "history": history,
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "best_weights": best_weights
    }

In [19]:
E0 = {"hidden_dims": [256]}
E1 = {"hidden_dims": [512, 256]}
E2 = {"hidden_dims": [512, 256], "dropout": 0.3}
E3 = {"hidden_dims": [512, 256], "batchnorm": True}

results = []

for config in [E0, E1, E2, E3]:
    res = run_experiment(config, epochs=15)
    results.append(res)
    print("Config:", config, "Best val acc:", res["best_val_acc"])

Config: {'hidden_dims': [256]} Best val acc: 0.9784166666666667
Config: {'hidden_dims': [512, 256]} Best val acc: 0.9779166666666667
Config: {'hidden_dims': [512, 256], 'dropout': 0.3} Best val acc: 0.982
Config: {'hidden_dims': [512, 256], 'batchnorm': True} Best val acc: 0.9816666666666667


In [20]:
best_exp_2_3 = max(results[2:], key=lambda x: x["best_val_acc"])
print("Best model config", best_exp_2_3["config"])

e4_res = run_experiment(
    best_exp_2_3["config"],
    epochs=30,
    early_stopping=True,
    patience=5
)
print("Config:", e4_res["config"], "Best val acc:", e4_res["best_val_acc"])
results.append(e4_res)

best_exp = max(results, key=lambda x: x["best_val_acc"])
print("Best config:", best_exp["config"])
print("Best val_acc:", best_exp["best_val_acc"])

Best model config {'hidden_dims': [512, 256], 'dropout': 0.3}
Config: {'hidden_dims': [512, 256], 'dropout': 0.3} Best val acc: 0.9805
Best config: {'hidden_dims': [512, 256], 'dropout': 0.3}
Best val_acc: 0.982


In [21]:
model = MLP(
    hidden_dims=best_exp["config"]["hidden_dims"],
    dropout=best_exp["config"].get("dropout", 0.0),
    batchnorm=best_exp["config"].get("batchnorm", False)
).to(device)

model.load_state_dict(best_exp["best_weights"])

test_loss, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss())
print("Final TEST accuracy:", test_acc)

Final TEST accuracy: 0.9821


In [28]:
DATASET_NAME = "KMNIST"

all_experiments = []

# E0-E3
for i, res in enumerate(results):
    exp_id = f"E{i}"
    
    model_summary = f"MLP hidden={res['config']['hidden_dims']}"
    
    row = {
        "experiment_id": exp_id,
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": model_summary,
        "dropout": res["config"].get("dropout", 0.0),
        "batchnorm": res["config"].get("batchnorm", False),
        "epochs_trained": len(res["history"]["val_acc"]),
        "best_val_accuracy": res["best_val_acc"],
        "best_val_loss": min(res["history"]["val_loss"])
    }
    
    all_experiments.append(row)

df_runs = pd.DataFrame(all_experiments)
df_runs.to_csv("artifacts/run.csv", index=False)

In [26]:
torch.save(best_exp["best_weights"], "artifacts/best_model.pt")

In [27]:
plt.figure(figsize=(10, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(best_exp["history"]["train_loss"], label="train_loss")
plt.plot(best_exp["history"]["val_loss"], label="val_loss")
plt.title("Loss")
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(best_exp["history"]["train_acc"], label="train_acc")
plt.plot(best_exp["history"]["val_acc"], label="val_acc")
plt.title("Accuracy")
plt.legend()

plt.tight_layout()

plt.savefig("artifacts/figures/curves_best.png")
plt.close()

print("Saved")

Saved:


In [31]:
best_config = {
    "dataset": DATASET_NAME,
    "seed": SEED,
    "hidden_dims": best_exp["config"]["hidden_dims"],
    "dropout": best_exp["config"].get("dropout", 0.0),
    "batchnorm": best_exp["config"].get("batchnorm", False),
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "loss": "CrossEntropyLoss",
    "early_stopping": False,
    "patience": 3
}

with open("artifacts/best_config.json", "w") as f:
    json.dump(best_config, f, indent=4)

print("Saved")

Saved
